# OBGPU Wavelet Comparison

This notebook compares any two saved `OBGPU` runs using the same wavelet-analysis pipeline as [LFP Wavelet Analysis.ipynb](/home/alek/OlfactoryBulb/notebooks/LFP%20Wavelet%20Analysis.ipynb).

It currently defaults to a matched `GammaSignature`, `1`-rank GPU pair that differs only in `legacy_parallel_dt`:

- `legacy_parallel_dt = False`
- `legacy_parallel_dt = True`

The summary includes actual runtime `dt`, total/run timing, raw and band-passed LFP drift, and full wavelet drift. If the run is too short for the old sniff-average calculation after skipping the first sniff, the notebook will say so instead of failing.


In [ ]:
# Optional: rerun this cell after editing RUN_A_DIR / RUN_B_DIR below.


In [ ]:
from pathlib import Path
import json
import pickle

import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.interpolate import interp1d
from scipy.signal import butter, lfilter
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-darkgrid")

ROOT = Path("/home/alek/OlfactoryBulb")
RUN_A_LABEL = "legacy_parallel_dt = False"
RUN_B_LABEL = "legacy_parallel_dt = True"
RUN_A_DIR = ROOT / "results/benchmarks/legacydt_false_r1_300ms_20260329_132413"
RUN_B_DIR = ROOT / "results/benchmarks/legacydt_true_r1_300ms_20260329_132521"

DT_MS = 0.1
SNIFF_DURATION_MS = 200
SKIP_FIRST_N_SNIFFS = 1
NUM_SNIFFS = 8
LOWCUT_HZ = 30
HIGHCUT_HZ = 120
WAVELET = "cgau5"
SCALE_LOW = 3
SCALE_HIGH = 32
N_SCALES = 50

def load_json(path):
    with open(path) as f:
        return json.load(f)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def interpolate(x, y, dt):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    f = interp1d(x, y, kind="linear")
    newx = np.arange(x.min(), x.max(), step=dt)
    newy = f(newx)
    return newx, newy

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return b, a

def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    return lfilter(b, a, data)

def trim_pair(a, b):
    n = min(len(a), len(b))
    return np.asarray(a)[:n], np.asarray(b)[:n]

def max_abs_diff(a, b):
    a, b = trim_pair(a, b)
    return float(np.max(np.abs(a - b))) if len(a) else 0.0

def load_wavelet_result(result_dir, dt=DT_MS, sniff_count=NUM_SNIFFS):
    summary = load_json(result_dir / "summary.json")
    t, lfp = load_pickle(result_dir / "lfp.pkl")
    t = np.asarray(t, dtype=float)
    lfp = np.asarray(lfp, dtype=float)
    t, lfp = interpolate(t, lfp, dt)

    fs_hz = 1 / dt * 1000
    lfp_bp = butter_bandpass_filter(lfp, LOWCUT_HZ, HIGHCUT_HZ, fs_hz, order=4)

    scales = np.linspace(SCALE_LOW / dt, SCALE_HIGH / dt, N_SCALES)
    cfs, frequencies = pywt.cwt(lfp_bp, scales, WAVELET, dt / 1000.0)
    wavelet_power = np.log(1 + np.abs(cfs))

    step = int(round(SNIFF_DURATION_MS / dt))
    chunks = []
    for i in range(SKIP_FIRST_N_SNIFFS, sniff_count + SKIP_FIRST_N_SNIFFS):
        start = i * step
        stop = (i + 1) * step - 2
        if stop <= wavelet_power.shape[1]:
            chunks.append(wavelet_power[:, start:stop])

    wavelet_power_average = sum(chunks) if chunks else None
    t_average = t[: wavelet_power_average.shape[1]] if wavelet_power_average is not None else None

    return {
        "summary": summary,
        "t": t,
        "lfp": lfp,
        "lfp_bp": lfp_bp,
        "frequencies": frequencies,
        "wavelet_power": wavelet_power,
        "wavelet_power_average": wavelet_power_average,
        "t_average": t_average,
        "sniff_avg_chunks": len(chunks),
    }


In [ ]:
run_a = load_wavelet_result(RUN_A_DIR)
run_b = load_wavelet_result(RUN_B_DIR)

t, lfp_a = trim_pair(run_a["t"], run_a["lfp"])
_, lfp_b = trim_pair(run_b["t"], run_b["lfp"])
_, lfp_bp_a = trim_pair(run_a["t"], run_a["lfp_bp"])
_, lfp_bp_b = trim_pair(run_b["t"], run_b["lfp_bp"])

common_cols = min(run_a["wavelet_power"].shape[1], run_b["wavelet_power"].shape[1])
full_power_diff = run_a["wavelet_power"][:, :common_cols] - run_b["wavelet_power"][:, :common_cols]

has_avg = run_a["wavelet_power_average"] is not None and run_b["wavelet_power_average"] is not None
if has_avg:
    avg_cols = min(run_a["wavelet_power_average"].shape[1], run_b["wavelet_power_average"].shape[1])
    avg_power_diff = run_a["wavelet_power_average"][:, :avg_cols] - run_b["wavelet_power_average"][:, :avg_cols]
else:
    avg_cols = 0
    avg_power_diff = None

summary_md = f"""
## Summary

| Metric | Value |
| --- | ---: |
| run A | `{RUN_A_LABEL}` |
| run B | `{RUN_B_LABEL}` |
| run A actual dt (ms) | {run_a['summary']['params']['actual_dt']:.3f} |
| run B actual dt (ms) | {run_b['summary']['params']['actual_dt']:.3f} |
| run A total time (s) | {run_a['summary']['timing_seconds']['total_max_rank']:.3f} |
| run B total time (s) | {run_b['summary']['timing_seconds']['total_max_rank']:.3f} |
| run B / run A total-time ratio | {run_b['summary']['timing_seconds']['total_max_rank'] / run_a['summary']['timing_seconds']['total_max_rank']:.3f}x |
| run A simulation time (s) | {run_a['summary']['timing_seconds']['run_max_rank']:.3f} |
| run B simulation time (s) | {run_b['summary']['timing_seconds']['run_max_rank']:.3f} |
| run B / run A simulation-time ratio | {run_b['summary']['timing_seconds']['run_max_rank'] / run_a['summary']['timing_seconds']['run_max_rank']:.3f}x |
| raw LFP max abs diff | {max_abs_diff(lfp_a, lfp_b):.3e} |
| band-passed LFP max abs diff | {max_abs_diff(lfp_bp_a, lfp_bp_b):.3e} |
| full wavelet-power max abs diff | {float(np.max(np.abs(full_power_diff))) if full_power_diff.size else 0.0:.3e} |
| sniff-average chunks in run A | {run_a['sniff_avg_chunks']} |
| sniff-average chunks in run B | {run_b['sniff_avg_chunks']} |
| sniff-averaged wavelet-power max abs diff | {('n/a (run too short after first-sniff skip)' if not has_avg else f'{float(np.max(np.abs(avg_power_diff))):.3e}')} |
"""

display(Markdown(summary_md))


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, constrained_layout=True)
axes[0].plot(t, lfp_a * 1000, lw=1.0, label=RUN_A_LABEL)
axes[0].plot(t, lfp_b * 1000, lw=1.0, alpha=0.8, label=RUN_B_LABEL)
axes[0].set_ylabel("Raw LFP x1000")
axes[0].set_title("Raw LFP comparison")
axes[0].legend(loc="best")
axes[1].plot(t, lfp_bp_a * 10000 - 200, lw=1.0, label=RUN_A_LABEL)
axes[1].plot(t, lfp_bp_b * 10000 - 200, lw=1.0, alpha=0.8, label=RUN_B_LABEL)
axes[1].set_ylabel("BP LFP x10000 - 200")
axes[1].set_xlabel("Simulation Time [ms]")
axes[1].set_title("Band-passed LFP comparison (30-120 Hz)")
axes[1].legend(loc="best")


In [ ]:
t_full = run_a["t"][: full_power_diff.shape[1]]
freq = run_a["frequencies"]

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True, constrained_layout=True)
axes[0].contourf(t_full, freq, run_a["wavelet_power"][:, : full_power_diff.shape[1]], 256, cmap="jet")
axes[0].set_ylim((20, 140))
axes[0].set_ylabel("Frequency [Hz]")
axes[0].set_title(f"{RUN_A_LABEL} wavelet power")

axes[1].contourf(t_full, freq, run_b["wavelet_power"][:, : full_power_diff.shape[1]], 256, cmap="jet")
axes[1].set_ylim((20, 140))
axes[1].set_ylabel("Frequency [Hz]")
axes[1].set_title(f"{RUN_B_LABEL} wavelet power")

amp = float(np.max(np.abs(full_power_diff))) if full_power_diff.size else 1.0
cf = axes[2].contourf(t_full, freq, full_power_diff, 256, cmap="coolwarm", vmin=-amp, vmax=amp)
axes[2].set_ylim((20, 140))
axes[2].set_ylabel("Frequency [Hz]")
axes[2].set_xlabel("Simulation Time [ms]")
axes[2].set_title(f"Wavelet power difference: {RUN_A_LABEL} - {RUN_B_LABEL}")
fig.colorbar(cf, ax=axes[2], label="Power difference")


In [ ]:
if not has_avg:
    display(Markdown(
        f"""
## Sniff-Average Note

This comparison pair is too short for the original sniff-average calculation after skipping the first sniff.

- run A chunks: `{run_a['sniff_avg_chunks']}`
- run B chunks: `{run_b['sniff_avg_chunks']}`

Use a longer run, such as the full `GammaSignature` `1800 ms`, if you want the sniff-averaged legacy wavelet panels.
"""
    ))
else:
    t_avg = run_a["t_average"][: avg_power_diff.shape[1]]

    fig, axes = plt.subplots(3, 1, figsize=(9, 12), sharex=True, constrained_layout=True)
    axes[0].contourf(t_avg, freq, run_a["wavelet_power_average"][:, : avg_power_diff.shape[1]], 256, cmap="jet")
    axes[0].set_ylim((20, 140))
    axes[0].set_ylabel("Frequency [Hz]")
    axes[0].set_title(f"{RUN_A_LABEL} sniff-averaged wavelet power")

    axes[1].contourf(t_avg, freq, run_b["wavelet_power_average"][:, : avg_power_diff.shape[1]], 256, cmap="jet")
    axes[1].set_ylim((20, 140))
    axes[1].set_ylabel("Frequency [Hz]")
    axes[1].set_title(f"{RUN_B_LABEL} sniff-averaged wavelet power")

    avg_amp = float(np.max(np.abs(avg_power_diff))) if avg_power_diff.size else 1.0
    cf = axes[2].contourf(t_avg, freq, avg_power_diff, 256, cmap="coolwarm", vmin=-avg_amp, vmax=avg_amp)
    axes[2].set_ylim((20, 140))
    axes[2].set_ylabel("Frequency [Hz]")
    axes[2].set_xlabel("Simulation Time within sniff [ms]")
    axes[2].set_title(f"Sniff-averaged wavelet power difference: {RUN_A_LABEL} - {RUN_B_LABEL}")
    fig.colorbar(cf, ax=axes[2], label="Power difference")
